# DX799 Week 11: DBSCAN, Hierarchical Agglomerative Clustering, and Dendrograms
**Integrated Capstone Project:** VFX, Animation, and the Film Industry  
**Research question:** Do DBSCAN and hierarchical clustering reveal groups of movies that K-Means missed, and what can these groups tell me about film characteristics and observed box-office outcomes?

In Week 10, I used K-Means to group movies based on information that could be available before release. This week, I will take a different approach. DBSCAN looks for dense groups and can label unusual movies as noise, while hierarchical agglomerative clustering (HAC) builds a nested grouping that can be examined using dendrograms. I will compare distance definitions, linkage methods, and cluster quality rather than assuming that whichever algorithm draws the clearest picture is necessarily the most useful.

## 1. Evaluation plan and safeguards

I will reuse the filtering rules from Weeks 8–10 and the same pre-release movie characteristics. The main cluster analysis will use the **10-feature compact design introduced in Week 10**: budget, runtime, release year, animation, four genre indicators, and cyclical sine/cosine values for release month. This avoids allowing 11 separately standardized month indicators to dominate the similarity calculation. The 19-feature Week 10 design remains a sensitivity comparison, not an identical feature representation.

I will use a **fixed random sample of up to 1,500 training movies** for DBSCAN and HAC so both methods see the same movies; this keeps pairwise-distance calculations practical. The training/test partition remains consistent with earlier weeks. I will choose settings from geometry alone, without using success labels, revenue, or other post-release outcomes. After choosing clusters, I will describe observed financial outcomes. This is an exploratory analysis, not a supervised forecast or a causal study.

The metrics have different meanings: DBSCAN reports **noise share** and cluster count as well as silhouette; HAC reports silhouette and cluster size. A high silhouette score can be misleading when DBSCAN labels most movies as noise, so I will report *coverage* alongside separation. I will also compare stability and interpretability before reaching conclusions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

SEED = 42
DATA_PATH = Path('movies.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('/mnt/data/movies.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Place the same movies.csv used in Weeks 8–10 alongside this notebook.')
raw = pd.read_csv(DATA_PATH)
print('Original data shape:', raw.shape)

## 2. Recreate the same movie sample

The original movie dataset includes post-release popularity and audience ratings, but these will **not** be clustering features. I only keep movies with positive reported budgets and revenues so I can later describe financial outcomes, which creates a selection limitation. My primary cluster inputs are observable movie characteristics, not a movie's actual performance. I retain the same 80/20 split with seed 42 and build the compact representation from training records only.

In [ ]:
df = raw.drop_duplicates(subset=['movie_id']).copy()
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
for column in ['budget', 'revenue', 'runtime']:
    df[column] = pd.to_numeric(df[column], errors='coerce')
df = df.loc[(df.budget > 0) & (df.revenue > 0)
            & df.release_year.between(1980, 2024)
            & df.runtime.between(40, 240)
            & df.release_date.notna()].copy()
df['budget_m'] = df.budget / 1e6
df['revenue_m'] = df.revenue / 1e6
df['gross_to_budget'] = df.revenue / df.budget
df['success'] = (df.gross_to_budget >= 2).astype(int)
genres = df.genres.fillna('').str.split('|')
df['animation'] = genres.map(lambda g: int('Animation' in g))
for genre in ['Action', 'Comedy', 'Drama', 'Horror']:
    df[f'genre_{genre.lower()}'] = genres.map(lambda g, name=genre: int(name in g))
df['release_month'] = df.release_date.dt.month
tr, te = train_test_split(np.arange(len(df)), test_size=.2, random_state=SEED)
train, test = df.iloc[tr].copy(), df.iloc[te].copy()
features = ['budget_m', 'runtime', 'release_year', 'animation',
            'genre_action', 'genre_comedy', 'genre_drama', 'genre_horror']
compact = train[features].copy()
compact['month_sin'] = np.sin(2*np.pi * train.release_month/12)
compact['month_cos'] = np.cos(2*np.pi * train.release_month/12)
assert compact.notna().sum().sum() == compact.size
scaler = StandardScaler()
X_all = scaler.fit_transform(compact)
rng = np.random.default_rng(SEED)
subset_positions = np.sort(rng.choice(len(train), size=min(1500, len(train)), replace=False))
X = X_all[subset_positions]
movies = train.iloc[subset_positions].copy().reset_index(drop=True)
print(f'Full sample: {len(df):,}; training: {len(train):,}; test: {len(test):,}')
print(f'Clustering sample: {len(movies):,} training movies, {X.shape[1]} inputs')
print('Features:', list(compact.columns))
print('Outcomes are held out from clustering and parameter selection.')

## 3. Exploratory analysis: what could form clusters?

Budget, runtime, and release date have different units. Without scaling, large numeric differences in budget or release year would overwhelm genre indicators. Standardizing them makes distances more comparable, but it also gives rare binary genre indicators considerable influence. Clusters may be sensitive to this representation. I will first look at the input distributions and then study the numerical cluster profiles.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(movies['budget_m'].clip(upper=250), bins=30)
ax[0].set(title='Production budgets (display capped at $250M)', xlabel='Budget ($M)', ylabel='Movies')
ax[1].hist(movies['runtime'], bins=25)
ax[1].set(title='Runtime distribution', xlabel='Minutes', ylabel='Movies')
plt.tight_layout(); plt.show()
print(movies[['budget_m','runtime','release_year']].describe().round(2).to_string())

## 4. DBSCAN: neighborhood radius, density, and noise

DBSCAN has two central parameters. **`eps`** is the radius around a movie within which other movies count as neighbors; **`min_samples`** is the number of observations (including the movie itself) required to form a core point. A smaller `eps` can isolate unusual movies but also leave much of the dataset unclustered. A larger `eps` can join groups and eventually collapse most movies into one cluster. I will use Euclidean distance initially and compare Manhattan distance as a sensitivity analysis.

A sorted distance-to-the-nearest-`min_samples`-th-neighbor plot can show a bend suggesting possible `eps` values, but it is a guide rather than an objectively correct solution.

In [ ]:
min_samples_values = [10, 20]
fig, ax = plt.subplots(figsize=(8, 4.5))
for m in min_samples_values:
    nn = NearestNeighbors(n_neighbors=m, metric='euclidean').fit(X)
    distances = nn.kneighbors(X)[0][:, -1]
    curve = np.sort(distances)
    ax.plot(np.arange(len(curve))/len(curve), curve, label=f'min_samples={m}')
ax.set(title='Sorted k-nearest-neighbor distances',
       xlabel='Fraction of movies (sorted)', ylabel='Distance to kth neighbor')
ax.grid(alpha=.2); ax.legend(); plt.tight_layout(); plt.show()
print('Euclidean k-distance quantiles:')
for m in min_samples_values:
    distances = NearestNeighbors(n_neighbors=m).fit(X).kneighbors(X)[0][:, -1]
    print(m, np.quantile(distances, [.50, .65, .75, .85, .90]).round(3))

## 5. Tune DBSCAN without looking at box-office outcomes

I will evaluate a small, reproducible grid of neighborhood radii derived from each distance metric's observed k-nearest-neighbor distances. I will report **number of clusters**, **noise fraction**, **largest-cluster share**, and **non-noise silhouette**. Silhouette is undefined for fewer than two non-noise clusters; in that case, I will record it as missing rather than manufacture a score. A solution is considered a *candidate* only if it produces at least two clusters, at least 50% coverage, and no cluster containing over 90% of the retained movies. Those thresholds are exploratory, not statistical guarantees.

To avoid selecting a solution that obtains a high silhouette by discarding most movies, I will rank valid candidates by silhouette multiplied by coverage. If none pass these rules, the conclusion will be that **DBSCAN did not find a satisfactory configuration among those tested**, not that a usable segmentation has been found.

In [ ]:
db_rows = []
db_models = {}
for metric in ['euclidean', 'manhattan']:
    for min_samples in min_samples_values:
        distances = NearestNeighbors(n_neighbors=min_samples, metric=metric).fit(X).kneighbors(X)[0][:, -1]
        eps_values = np.unique(np.quantile(distances, [.55, .65, .75, .85, .90]).round(4))
        for eps in eps_values:
            labels = DBSCAN(eps=float(eps), min_samples=min_samples, metric=metric).fit_predict(X)
            valid = labels != -1
            ids, counts = np.unique(labels[valid], return_counts=True)
            n_clusters = len(ids)
            coverage = valid.mean()
            largest = counts.max()/valid.sum() if valid.any() else np.nan
            sil = (silhouette_score(X[valid], labels[valid], metric=metric,
                                    sample_size=min(800, int(valid.sum())), random_state=SEED)
                   if n_clusters >= 2 and valid.sum() > n_clusters else np.nan)
            suitable = n_clusters >= 2 and coverage >= .50 and largest <= .90
            db_rows.append(dict(metric=metric, min_samples=min_samples, eps=float(eps),
                                clusters=n_clusters, noise_share=1-coverage, coverage=coverage,
                                largest_share=largest, silhouette=sil,
                                suitable=suitable,
                                adjusted_score=(sil*coverage if suitable else np.nan)))
            db_models[(metric, min_samples, float(eps))] = labels
db_results = pd.DataFrame(db_rows)
print(db_results.sort_values(['suitable','adjusted_score'], ascending=[False,False],na_position='last')
      .head(15).round(4).to_string(index=False))
valid_options = db_results.loc[db_results.suitable].sort_values('adjusted_score',ascending=False)
if len(valid_options):
    db_choice = valid_options.iloc[0]
    db_labels = db_models[(db_choice.metric, int(db_choice.min_samples), float(db_choice.eps))]
    print('Selected DBSCAN candidate:', db_choice[['metric','min_samples','eps','clusters',
          'noise_share','silhouette','adjusted_score']].to_dict())
else:
    db_choice, db_labels = None, None
    print('No DBSCAN configuration met the predeclared coverage and separation requirements.')

## 6. DBSCAN sensitivity and noise interpretation

Comparing Manhattan with Euclidean distance does not mean that either is universally better. In this dataset, changing the metric changes which combinations of budget, genre, runtime, and season are treated as nearby. I will graph silhouette and noise share together so that the tradeoff is explicit. When the selected solution contains noise, I will examine which types of movies were excluded, since those cases might be especially interesting for the film industry.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
for metric in ['euclidean','manhattan']:
    d = db_results[(db_results.metric == metric) & (db_results.min_samples == 10)].sort_values('eps')
    ax[0].plot(d.eps, d.noise_share, marker='o', label=metric)
    ax[1].plot(d.eps, d.silhouette, marker='o', label=metric)
ax[0].set(title='Noise fraction versus eps', xlabel='eps', ylabel='Share labeled noise')
ax[1].set(title='Non-noise silhouette versus eps', xlabel='eps', ylabel='Silhouette (when defined)')
for a in ax: a.grid(alpha=.2); a.legend()
plt.tight_layout(); plt.show()
if db_labels is not None:
    movies['dbscan_cluster'] = db_labels
    counts = movies.dbscan_cluster.value_counts().sort_index()
    print('Selected cluster sizes (-1 = noise): ' , counts.to_string())
    print('Descriptive budget and observed outcome profiles (not used to choose clusters):')
    print(movies.groupby('dbscan_cluster').agg(n=('movie_id','size'),median_budget_m=('budget_m','median'),
        median_runtime=('runtime','median'),animation_share=('animation','mean'),
        success_share=('success','mean'),median_revenue_m=('revenue_m','median')).round(3).to_string())
else:
    print('DBSCAN profiling omitted because no acceptable configuration was found.')

## 7. Hierarchical clustering and linkage methods

HAC starts with each movie as its own cluster and repeatedly merges groups. **Single linkage** uses the nearest pair of points and can make long chains. **Complete linkage** uses the farthest pair and tends to produce tighter groups. **Average linkage** compares average inter-cluster distance. **Ward linkage** merges groups based on the increase in within-cluster variance and requires Euclidean geometry. These methods can give different groups even when applied to identical scaled features.

I will compare 2–8 clusters for each linkage method using the same 1,500 training movies and Euclidean distance, so the linkage comparison is controlled. Silhouette scores and cluster-size balance will help us evaluate the results, but I will also check whether the groups make sense in the film context.

In [ ]:
hac_rows = []
hac_models = {}
for linkage_method in ['single','complete','average','ward']:
    for k in range(2, 9):
        model = AgglomerativeClustering(n_clusters=k, linkage=linkage_method, metric='euclidean')
        labels = model.fit_predict(X)
        sizes = np.bincount(labels, minlength=k)
        score = silhouette_score(X, labels, metric='euclidean', sample_size=min(1000,len(X)),random_state=SEED)
        hac_rows.append({'linkage':linkage_method,'k':k,'silhouette':score,
                         'smallest_cluster':int(sizes.min()), 'largest_share':float(sizes.max()/len(labels))})
        hac_models[(linkage_method,k)] = labels
hac_results = pd.DataFrame(hac_rows)
print(hac_results.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(9, 4.5))
for method in ['single','complete','average','ward']:
    d = hac_results[hac_results.linkage == method]
    ax.plot(d.k, d.silhouette, marker='o', label=method)
ax.set(title='Hierarchical clustering: silhouette by linkage and k',
       xlabel='Number of clusters',ylabel='Sampled Euclidean silhouette')
ax.grid(alpha=.2); ax.legend(); plt.tight_layout(); plt.show()

## 8. Select and interpret HAC clusters

For a defensible comparison, I will prefer solutions whose largest cluster holds no more than 90% of the sample and whose smallest cluster contains at least 2% of movies. These rules help prevent a very high silhouette score from winning simply because the algorithm isolated a tiny group. I will select the highest-silhouette configuration **among eligible options**, without looking at financial success. If none qualify, I will report that limitation rather than force an artificial selection.

In [ ]:
eligible = hac_results[(hac_results.largest_share <= .90)
                       & (hac_results.smallest_cluster >= max(2, int(.02*len(X))))]
if eligible.empty:
    hac_choice, hac_labels = None, None
    print('No HAC setting met the predeclared cluster-balance rules.')
else:
    hac_choice = eligible.sort_values('silhouette', ascending=False).iloc[0]
    hac_labels = hac_models[(hac_choice.linkage, int(hac_choice.k))]
    movies['hac_cluster'] = hac_labels
    print('Selected HAC configuration:', hac_choice.to_dict())
    hac_profile = movies.groupby('hac_cluster').agg(
        movies=('movie_id','size'),median_budget_m=('budget_m','median'),
        median_runtime=('runtime','median'),mean_release_year=('release_year','mean'),
        action_share=('genre_action','mean'),comedy_share=('genre_comedy','mean'),
        horror_share=('genre_horror','mean'),animation_share=('animation','mean'),
        success_share=('success','mean'),median_revenue_m=('revenue_m','median'),
        median_gross_to_budget=('gross_to_budget','median'))
    print('Movie profiles (financial outcomes only for post-hoc description):')
    print(hac_profile.round(3).to_string())

## 9. Dendrograms: understanding how clusters merge

A dendrogram shows which movies or clusters merge as the algorithm proceeds. The vertical height represents the distance or linkage criterion at which they combine; it is **not** a probability or an accuracy score. I will use a fixed 100-movie subset solely to make the dendrogram legible. Since it is a subset, the exact branch structure will differ from fitting HAC to the full 1,500 observations. It illustrates how the linkage rules work, not the precise grouping of every movie.

In [ ]:
dendro_idx = np.sort(rng.choice(len(X),size=min(100,len(X)),replace=False))
X_dendro = X[dendro_idx]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax,method in zip(axes.ravel(), ['single','complete','average','ward']):
    z = linkage(X_dendro, method=method, metric='euclidean')
    dendrogram(z, ax=ax, no_labels=True, color_threshold=0)
    ax.set(title=f'{method.capitalize()} linkage', xlabel='Movies (100 sampled)',
           ylabel='Merge height')
plt.tight_layout(); plt.show()
print('These 100-movie dendrograms illustrate the effect of linkage rules; model tuning above uses 1,500 movies.')

## 10. Comparing DBSCAN, HAC, and the Week 10 K-Means approach

K-Means prespecifies the number of clusters and minimizes squared Euclidean distance to centroids. HAC also groups all movies, but its nested merges depend heavily on linkage. DBSCAN does not require specifying a target cluster count and may treat some movies as noise. The methods therefore answer related but different questions. I will fit a K-Means benchmark with the HAC-selected number of groups on the **same subset**, then compare silhouette and cluster balance. If HAC has no eligible configuration, I will use k=3 only as an illustrative benchmark and label it as such, not as Week 10's selected answer. The actual Week 10 selected k must be taken from the executed Week 10 notebook.

Cluster IDs are arbitrary and cannot be compared directly. When DBSCAN labels many points as noise, adjusted Rand index also reflects that noise-label assignment, so I will interpret it cautiously.

In [ ]:
baseline_k = int(hac_choice.k) if hac_choice is not None else 3
km = KMeans(n_clusters=baseline_k,n_init=10,random_state=SEED)
km_labels = km.fit_predict(X)
summary_rows = [{
    'Method':f'K-Means (k={baseline_k})', 'Clusters':baseline_k, 'Coverage':1.0,
    'Silhouette':silhouette_score(X,km_labels,metric='euclidean',sample_size=min(1000,len(X)),random_state=SEED)
}]
if hac_labels is not None:
    summary_rows.append({'Method':f'HAC ({hac_choice.linkage}, k={int(hac_choice.k)})',
      'Clusters':int(hac_choice.k),'Coverage':1.0,'Silhouette':float(hac_choice.silhouette)})
if db_labels is not None:
    mask=db_labels!=-1
    summary_rows.append({'Method':f'DBSCAN ({db_choice.metric})','Clusters':int(db_choice.clusters),
      'Coverage':float(mask.mean()), 'Silhouette':float(db_choice.silhouette)})
comparison=pd.DataFrame(summary_rows)
print(comparison.round(4).to_string(index=False))
if hac_labels is not None:
    print('K-Means versus HAC adjusted Rand index:',round(adjusted_rand_score(km_labels,hac_labels),4))
if db_labels is not None and hac_labels is not None:
    print('HAC versus DBSCAN (non-noise subset) adjusted Rand index:',
          round(adjusted_rand_score(hac_labels[db_labels!=-1],db_labels[db_labels!=-1]),4))

## 11. Stability check and two-dimensional visualization

Clustering can be unstable when the sample changes. I will compare the selected HAC method between two overlapping 80% subsamples by examining their assignments **only on the movies shared by both samples**. This produces a sensitivity diagnostic, not a confidence interval. For DBSCAN, I will report the noise fraction and cluster count under a small ±10% change to `eps` rather than assuming the chosen radius is precisely correct.

PCA will provide a two-dimensional picture, but the actual clustering uses all ten standardized inputs. A visually compact PCA group does not prove that the full-dimensional clusters are meaningful.

In [ ]:
if hac_choice is not None:
    rng2=np.random.default_rng(SEED)
    a=np.sort(rng2.choice(len(X),size=int(.8*len(X)),replace=False))
    b=np.sort(rng2.choice(len(X),size=int(.8*len(X)),replace=False))
    shared, ia, ib=np.intersect1d(a,b,return_indices=True)
    def hac_fit(z):
        return AgglomerativeClustering(n_clusters=int(hac_choice.k),
                 linkage=hac_choice.linkage,metric='euclidean').fit_predict(z)
    la=hac_fit(X[a]); lb=hac_fit(X[b])
    print('HAC overlapping-subsample ARI:',round(adjusted_rand_score(la[ia],lb[ib]),4),
          '| Shared movies:',len(shared))
if db_choice is not None:
    for multiplier in [.9,1,1.1]:
        l=DBSCAN(eps=float(db_choice.eps)*multiplier, min_samples=int(db_choice.min_samples),
                 metric=db_choice.metric).fit_predict(X)
        print(f'DBSCAN eps x{multiplier:.1f}: clusters={len(set(l)-{-1})}, noise={np.mean(l==-1):.1%}')
pca=PCA(n_components=2,random_state=SEED)
xy=pca.fit_transform(X)
fig, ax=plt.subplots(1,2,figsize=(12,4.6))
ax[0].scatter(xy[:,0],xy[:,1],c=km_labels,s=11,alpha=.6,cmap='tab10')
ax[0].set(title='K-Means groups (PCA display)',xlabel='PC1',ylabel='PC2')
show_labels = db_labels if db_labels is not None else hac_labels
if show_labels is not None:
    ax[1].scatter(xy[:,0],xy[:,1],c=show_labels,s=11,alpha=.6,cmap='tab10')
    ax[1].set(title=('DBSCAN groups and noise' if db_labels is not None else 'HAC groups')+' (PCA display)',
              xlabel='PC1',ylabel='PC2')
else:
    ax[1].text(.5,.5,'No eligible DBSCAN or HAC result',ha='center',va='center',transform=ax[1].transAxes)
    ax[1].set_axis_off()
plt.tight_layout();plt.show()
print('Variance represented by first two PCs:', round(pca.explained_variance_ratio_.sum(),4))

## 12. Findings to use in Milestone Two

**What I will compare:** DBSCAN cluster count and noise coverage at different `eps` values, whether Euclidean or Manhattan distance leads to more useful grouping, which HAC linkage method has the best balance between silhouette and cluster sizes, and how much the solution changes when the sample or hyperparameters change. These are concrete grounds for explaining tuning and model limitations rather than merely naming the algorithms.

**Expected patterns:** I expect budget and genres to affect movie similarity. I also expect a dense-clustering method to struggle with the mix of continuous and binary predictors. Both are *hypotheses* until I run the notebook; they are not conclusions.

**What counts as a meaningful result:** A plausible set of groups, adequate coverage, reasonable silhouette values, and recognizable differences in movie characteristics. Differences in post-release success rates may be interesting, but they do not prove that a cluster *caused* success or that a studio can make dependable revenue predictions from it.

**Limits:** The analysis only represents films with positive reported budgets and revenues from 1980–2024. Reported gross revenue is not profit. A random sample of 1,500 movies represents the filtered training set, not a forecast for future releases. In high-dimensional spaces, distance metrics can lose discriminating power; noisy or weakly separated clusters may be the honest result. The final Milestone Two write-up should cite the **executed** results and figures, not unverified expectations.